# Artificial Intelligence & Machine Learning - Task 2

## Feature Engineering, Model Optimization & Performance Comparison

### House Price Prediction using California Housing Dataset

**Submitted by:** Aman Raj

**Internship:** Sumerix Global

---

### Objective

The objective of this project is to improve the basic House Price Prediction model developed in Task 1 by applying feature scaling, training multiple regression algorithms, comparing their performance, and selecting the best-performing model based on evaluation metrics.

In [ ]:
# =====================================================
# Import Required Libraries
# =====================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge

from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

print("Libraries Imported Successfully!")

In [ ]:
# =====================================================
# Load California Housing Dataset
# =====================================================

housing = fetch_california_housing(as_frame=True)

df = pd.concat(
    [
        housing.data,
        housing.target.rename("HousePrice")
    ],
    axis=1
)

print("Dataset Loaded Successfully!")

df.head()

In [ ]:
# =====================================================
# Dataset Information
# =====================================================

print("Shape of Dataset:")
print(df.shape)

print("\n")

print("Column Names:")
print(df.columns)

print("\n")

print("Missing Values:")
print(df.isnull().sum())

print("\n")

print("Statistical Summary:")

df.describe()

In [ ]:
# =====================================================
# Dataset Information
# =====================================================

df.info()

In [ ]:
# =====================================================
# Feature Distribution
# =====================================================

df.hist(
    figsize=(15,10),
    bins=30
)

plt.suptitle("Feature Distributions")

plt.tight_layout()

plt.show()

In [ ]:
# =====================================================
# Correlation Heatmap
# =====================================================

plt.figure(figsize=(12,8))

sns.heatmap(
    df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
# =====================================================
# Separate Features and Target Variable
# =====================================================

X = df.drop("HousePrice", axis=1)

y = df["HousePrice"]

print("Features Shape:", X.shape)

print("Target Shape:", y.shape)

In [ ]:
# =====================================================
# Feature Scaling
# =====================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Feature Scaling Completed!")

In [ ]:
# =====================================================
# Train Test Split
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples :", X_train.shape)

print("Testing Samples :", X_test.shape)

In [ ]:
# =====================================================
# Create Multiple Machine Learning Models
# =====================================================

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=5,
        random_state=42
    )
}

print("Models Created Successfully!")

In [ ]:
# =====================================================
# Train Models and Compare Performance
# =====================================================

results = []

for name, model in models.items():

    # Train Model
    model.fit(X_train, y_train)

    # Predict
    predictions = model.predict(X_test)

    # Evaluation Metrics
    mae = mean_absolute_error(y_test, predictions)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    r2 = r2_score(y_test, predictions)

    results.append([
        name,
        mae,
        rmse,
        r2
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R2 Score"
    ]
)

results_df

In [ ]:
# =====================================================
# Sort Models by Best R² Score
# =====================================================

results_df = results_df.sort_values(
    by="R2 Score",
    ascending=False
)

results_df

In [ ]:
# =====================================================
# Model Comparison (R² Score)
# =====================================================

plt.figure(figsize=(8,5))

sns.barplot(
    data=results_df,
    x="Model",
    y="R2 Score"
)

plt.title("Model Comparison (R² Score)")

plt.xticks(rotation=10)

plt.show()

In [ ]:
# =====================================================
# RMSE Comparison
# =====================================================

plt.figure(figsize=(8,5))

sns.barplot(
    data=results_df,
    x="Model",
    y="RMSE"
)

plt.title("Model Comparison (RMSE)")

plt.xticks(rotation=10)

plt.show()

In [ ]:
# =====================================================
# Select Best Model Automatically
# =====================================================

best_model_name = results_df.iloc[0]["Model"]

print("Best Model:")

print(best_model_name)

In [ ]:
# =====================================================
# Train Best Model
# =====================================================

best_model = models[best_model_name]

best_model.fit(
    X_train,
    y_train
)

y_pred = best_model.predict(X_test)

print("Best Model Trained Successfully!")

In [ ]:
# =====================================================
# Actual vs Predicted
# =====================================================

plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.5
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red",
    linewidth=2
)

plt.xlabel("Actual House Prices")

plt.ylabel("Predicted House Prices")

plt.title(f"Actual vs Predicted ({best_model_name})")

plt.show()

In [ ]:
# =====================================================
# Residual Plot
# =====================================================

residuals = y_test - y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals,
    alpha=0.5
)

plt.axhline(
    y=0,
    color="red",
    linestyle="--"
)

plt.xlabel("Predicted Values")

plt.ylabel("Residuals")

plt.title("Residual Plot")

plt.show()

In [ ]:
# =====================================================
# Feature Importance
# =====================================================

if best_model_name == "Decision Tree":

    importance = pd.DataFrame({
        "Feature": X.columns,
        "Importance": best_model.feature_importances_
    })

    importance = importance.sort_values(
        by="Importance",
        ascending=False
    )

    plt.figure(figsize=(10,6))

    sns.barplot(
        data=importance,
        x="Importance",
        y="Feature"
    )

    plt.title("Feature Importance")

    plt.show()

    importance

In [ ]:
# =====================================================
# Save Best Model
# =====================================================

joblib.dump(
    best_model,
    "best_house_price_model.pkl"
)

print("Best model saved successfully!")

In [ ]:
# =====================================================
# Load Saved Model
# =====================================================

loaded_model = joblib.load(
    "best_house_price_model.pkl"
)

print("Model Loaded Successfully!")

In [ ]:
# =====================================================
# Predict New House Price
# =====================================================

sample_house = [[
    8.3252,
    41,
    6.984127,
    1.023810,
    322,
    2.555556,
    37.88,
    -122.23
]]

# Apply the same scaling used during training
sample_house_scaled = scaler.transform(sample_house)

prediction = loaded_model.predict(sample_house_scaled)

print("Predicted House Price:")

print(prediction[0])

In [ ]:
# =====================================================
# Final Model Performance
# =====================================================

print(results_df)

# Conclusion

In this project, feature scaling was applied using StandardScaler to improve model performance.

Three regression algorithms were trained and evaluated:

- Linear Regression
- Ridge Regression
- Decision Tree Regressor

Each model was compared using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² Score

The best-performing model was selected based on evaluation metrics.

This project demonstrates a complete machine learning workflow involving preprocessing, model optimization, performance comparison, visualization, and model persistence.

# Future Improvements

Future enhancements to this project include:

- Hyperparameter tuning using GridSearchCV
- Random Forest Regression
- XGBoost Regression
- Feature Engineering
- Cross Validation
- Deployment using Flask or Streamlit